In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
pd.set_option("display.max_column",500)
pd.set_option("display.max_row",500)

In [4]:
# To read the second sheet (index 1)
private_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Schools, Classes & Staff by Pro")
private_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Schools, Classes & Staff by Pro")
private_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Schools, Classes & Staff by Pro")

In [5]:
row =private_18_19.iloc[0]
private_18_19 = private_18_19.rename(columns=row)
private_18_19 = private_18_19.iloc[2:-3]
private_18_19 = private_18_19.reset_index(drop=True)
private_18_19 = private_18_19.set_index("Province")
cols = list(private_18_19.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[10] = "Foreigner Teaching Staff(F)"
private_18_19.columns = cols
private_18_19 = private_18_19.astype('Int64')

In [6]:
private_18_19.shape

(25, 14)

In [7]:
private_19_20 =private_19_20.iloc[1:-3]
private_19_20 = private_19_20.reset_index(drop=True)
private_19_20 = private_19_20.set_index("Province")
cols = list(private_19_20.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[10] = "Foreigner Teaching Staff(F)"
private_19_20.columns = cols
private_19_20 = private_19_20.astype('Int64')

In [8]:
row =private_20_21.iloc[0]
private_20_21 = private_20_21.rename(columns=row)
private_20_21 = private_20_21.iloc[2:-3]
private_20_21 = private_20_21.reset_index(drop=True)
private_20_21 = private_20_21.set_index("Province")
cols = list(private_20_21.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[9] = "Foreigner Teaching Staff(F)"
private_20_21.columns = cols
private_20_21 = private_20_21.astype('Int64')
private_20_21.shape

(25, 13)

In [9]:
private_18_19.insert(0,'year',2018)
private_19_20.insert(0,'year',2019)
private_20_21.insert(0,'year',2020)

In [10]:
combined_df = pd.concat([private_18_19, private_19_20, private_20_21])

In [11]:
# 1. Define the mapping dictionary
rename_map = {
    'Number of\nSchools': 'Schools',
    'Number of\nClasses': 'Classes',
    'Enrollment': 'Enrollment_Total',
    'Enrollment(F)': 'Enrollment_Girl',
    'Teaching Staff': 'TeachStaff_Total',
    'Teaching Staff(F)': 'TeachStaff_Female',
    'Non-Teaching Staff': 'NonTeachStaff_Total',
    'Non-Teaching Staff(F)': 'NonTeachStaff_Female',
    'Govern. staff': 'Govern_Staff',
    'Foreigner Teaching Staff': 'Foreigner_Staff_Total',
    'Foreigner Teaching Staff(F)': 'Foreigner_Staff_Female',
    'Buildings': 'Buildings',
    'Rooms': 'Rooms',
    'Classrooms': 'Classrooms'
}

# 2. Apply the rename to your combined dataframe
# errors='ignore' ensures it won't crash if a column is already renamed
combined_df = combined_df.rename(columns=rename_map, errors='ignore')

# 3. Double check the result
print(combined_df.columns.tolist())

['year', 'Schools', 'Classes', 'Enrollment_Total', 'Enrollment_Girl', 'TeachStaff_Total', 'TeachStaff_Female', 'NonTeachStaff_Total', 'NonTeachStaff_Female', 'Govern_Staff', 'Foreigner_Staff_Total', 'Foreigner_Staff_Female', 'Buildings', 'Rooms', 'Classrooms']


In [12]:
combined_df.columns

Index(['year', 'Schools', 'Classes', 'Enrollment_Total', 'Enrollment_Girl',
       'TeachStaff_Total', 'TeachStaff_Female', 'NonTeachStaff_Total',
       'NonTeachStaff_Female', 'Govern_Staff', 'Foreigner_Staff_Total',
       'Foreigner_Staff_Female', 'Buildings', 'Rooms', 'Classrooms'],
      dtype='object')

In [13]:
def set_province_index(df):
    """Rename the first column to Province, drop junk rows, set as index."""
    df = df.rename(columns={df.columns[0]: "Province"})
    # Drop rows where Province is NaN (header-junk rows 0 and 1)
    df = df[df["Province"].notna()].copy()
    df = df.reset_index(drop=True)
    df = df.set_index("Province")
    return df


# ═══════════════════════════════════════════════════════════════
# 1.  ENROLLMENT & REPEATERS  (Enroll_and_repeat_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════

Enroll_and_repeat_18_19 = pd.read_excel(
    'organized_education_data/private_schools_2018-2019.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)
Enroll_and_repeat_19_20 = pd.read_excel(
    'organized_education_data/private_schools_2019-2020.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)
Enroll_and_repeat_20_21 = pd.read_excel(
    'organized_education_data/private_schools_2020-2021.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)

# Build clean column names (same for all three years)
sub_headers = ['Enrollment_Total', 'Enrollment_Girl', 'Repeaters_Total', 'Repeaters_Girl']
enroll_cols = ['Province']
for i in range(1, 13):
    for sub in sub_headers:
        enroll_cols.append(f'Grade_{i}_{sub}')
for sub in sub_headers:
    enroll_cols.append(f'Total_{sub}')


def clean_enroll(df):
    df.columns = enroll_cols
    # 1. Drop the junk sub-header rows
    df = df[df["Province"].notna()].copy()
    
    # 2. REMOVE FOOTERS: Filter out aggregate rows
    # This is more robust than [:-3] because it looks for the names specifically
    df = df[~df["Province"].astype(str).str.contains("Whole Kingdom|Area", case=False, na=False)]
    
    df = df.reset_index(drop=True)
    df = df.set_index("Province")
    
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df


Enroll_and_repeat_18_19 = clean_enroll(Enroll_and_repeat_18_19)
Enroll_and_repeat_19_20 = clean_enroll(Enroll_and_repeat_19_20)
Enroll_and_repeat_20_21 = clean_enroll(Enroll_and_repeat_20_21)

# Add Year and combine
Enroll_and_repeat_18_19['Year'] = 2018
Enroll_and_repeat_19_20['Year'] = 2019
Enroll_and_repeat_20_21['Year'] = 2020

combined_enroll = pd.concat(
    [Enroll_and_repeat_18_19, Enroll_and_repeat_19_20, Enroll_and_repeat_20_21]
)
# Move Year to front
cols = list(combined_enroll.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_enroll = combined_enroll[cols]


# ═══════════════════════════════════════════════════════════════
# 2.  TEACHING STAFF BY EDUCATION LEVEL (Staff_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════

Staff_18_19 = pd.read_excel(
    'organized_education_data/private_schools_2018-2019.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)
Staff_19_20 = pd.read_excel(
    'organized_education_data/private_schools_2019-2020.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)
Staff_20_21 = pd.read_excel(
    'organized_education_data/private_schools_2020-2021.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)

# Column structure (19 columns):
# Province | Total: Primary L.sec U.Sec Graduate PostGrad PhD
#           | Female: Primary L.sec U.Sec Graduate PostGrad PhD
#           | Without Ped Training: Primary L.sec U.Sec Graduate PostGrad PhD
staff_cols = ['Province',
    'Total_Primary', 'Total_LSec', 'Total_USec', 'Total_Graduate', 'Total_PostGrad', 'Total_PhD',
    'Female_Primary', 'Female_LSec', 'Female_USec', 'Female_Graduate', 'Female_PostGrad', 'Female_PhD',
    'NoPedTrain_Primary', 'NoPedTrain_LSec', 'NoPedTrain_USec',
    'NoPedTrain_Graduate', 'NoPedTrain_PostGrad', 'NoPedTrain_PhD'
]


def clean_staff(df):
    # Skip row 0 (headers) and skip the last 3 rows (footers)
    df = df.iloc[1:-3].copy() 
    
    df.columns = staff_cols
    df = df[df["Province"].notna()].copy()
    df = df.set_index("Province")
    
    # Optional: double check to remove any remaining "Total" rows
    df = df[~df.index.astype(str).str.contains("Total|Whole", case=False, na=False)]
    
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df


Staff_18_19 = clean_staff(Staff_18_19)
Staff_19_20 = clean_staff(Staff_19_20)
Staff_20_21 = clean_staff(Staff_20_21)

# Add Year and combine
Staff_18_19['Year'] = 2018
Staff_19_20['Year'] = 2019
Staff_20_21['Year'] = 2020

combined_staff = pd.concat([Staff_18_19, Staff_19_20, Staff_20_21])
cols = list(combined_staff.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_staff = combined_staff[cols]

In [50]:
combined_staff.columns

Index(['Total_Primary', 'Total_LSec', 'Total_USec', 'Total_Graduate',
       'Total_PostGrad', 'Total_PhD', 'Female_Primary', 'Female_LSec',
       'Female_USec', 'Female_Graduate', 'Female_PostGrad', 'Female_PhD',
       'NoPedTrain_Primary', 'NoPedTrain_LSec', 'NoPedTrain_USec',
       'NoPedTrain_Graduate', 'NoPedTrain_PostGrad', 'NoPedTrain_PhD'],
      dtype='object')

In [15]:
private_GER_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Access & Admission Rate Indicat")
private_GER_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Access & Admission Rate Indicat")
private_GER_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Access & Admission Rate Indicat")

In [16]:
cols = ["Province","GAR(Total)","GAR(F)","GAR(M)","NAR(Total)","NAR(F)","NAR(M)","GER pri","GER L.Sec","GER U.Sec","GER pri(F)","GER L.Sec(F)","GER U.Sec(F)","GER pri(M)","GER L.Sec(M)","GER U.Sec(M)","NER pri","NER L.Sec","NER U.Sec","NER pri(F)","NER L.Sec(F)","NER U.Sec(F)","NER pri(M)","NER L.Sec(M)","NER U.Sec(M)"]
private_GER_18_19.columns = cols
private_GER_18_19 = private_GER_18_19[2:-3]
private_GER_18_19 = private_GER_18_19.set_index("Province")
private_GER_18_19 = private_GER_18_19.astype('float64')

In [17]:
private_GER_19_20.columns = cols
private_GER_19_20 = private_GER_19_20[2:-3]
private_GER_19_20 = private_GER_19_20.set_index("Province")
private_GER_19_20 = private_GER_19_20.astype('float64')

In [18]:
private_GER_20_21.columns = cols
private_GER_20_21 = private_GER_20_21[3:-3]
private_GER_20_21 = private_GER_20_21.set_index("Province")
private_GER_20_21 = private_GER_20_21.astype('float64')

In [19]:
private_GER_18_19.insert(0,'year',2018)
private_GER_19_20.insert(0,'year',2019)
private_GER_20_21.insert(0,'year',2020)

In [20]:
GER = pd.concat([private_GER_18_19,private_GER_19_20,private_GER_20_21])

In [21]:
private_COM_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Completion Rate 2018-2019")
private_COM_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Completion Rate 2019-2020")
private_COM_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Completion Rate 2021-2022")

In [22]:
com_col = ['Province','pri(Total)','pri(F)','pri(M)','L.sec(Total)','L.sec(F)','L.sec(M)','U.sec(Total)','U.sec(F)','U.sec(M)']
private_COM_18_19.columns = com_col
private_COM_18_19 = private_COM_18_19[1:-3]
private_COM_18_19 = private_COM_18_19.set_index("Province")
private_COM_18_19 = private_COM_18_19.astype('float64')

In [23]:
private_COM_19_20.columns = com_col
private_COM_19_20 = private_COM_19_20[1:-3]
private_COM_19_20 = private_COM_19_20.set_index("Province")
private_COM_19_20 = private_COM_19_20.astype('float64')

In [24]:
private_COM_20_21.columns = com_col
private_COM_20_21 = private_COM_20_21[2:-3]
private_COM_20_21 = private_COM_20_21.set_index("Province")
private_COM_20_21 = private_COM_20_21.astype('float64')

In [25]:
private_COM_18_19.insert(0,'year',2018)
private_COM_19_20.insert(0,'year',2019)
private_COM_20_21.insert(0,'year',2020)

In [26]:
COM = pd.concat([private_COM_18_19,private_COM_19_20,private_COM_20_21])

In [27]:
private_ENR_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Enrollment by Level 2018-2019")
private_ENR_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Enrollment by Level 2019-2020")
private_ENR_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Enrollment by Level 2020-2021")

In [28]:
private_ENR_18_19.head()

,"Table 1: Schools, Classes, Students and Staff",Number of\nSchools,Pre-School,Unnamed: 3,Primary,Unnamed: 5,Lower Sec.,Unnamed: 7,Upper Sec.,Unnamed: 9,Total,Unnamed: 11,%Female,Unnamed: 13
0,NaN,NaN,Total,Girl,Total,Girl,Total,Girl,Total,Girl,Total,Girl,Primary,All Level
1,Banteay Meanchey,121.0,3393,1653,14332,6879,2917,1421,1300,658,21942,10611,48,48.4
2,Battambang,45.0,1273,605,4348,2139,1661,813,1522,814,8804,4371,49.2,49.6
3,Kampong Cham,25.0,853,455,2532,1213,407,192,218,109,4010,1969,47.9,49.1
4,Kampong Chhnang,25.0,1532,736,1905,905,608,230,460,184,4505,2055,47.5,45.6


In [29]:
# 1. Include 'Province' at the start (Total of 14 columns)
ENR_COL = [
    'Province', 'Number of school', 'pre(Total)', 'pre(F)', 
    'pri(Total)', 'pri(F)', 'L.sec(Total)', 'L.sec(F)', 
    'U.sec(Total)', 'U.sec(F)', 'Total(Total)', 'Total(F)', 
    'Female pri(%)', 'Female all(%)'
]
private_ENR_18_19.columns = ENR_COL
private_ENR_18_19 = private_ENR_18_19.iloc[1:-3].copy()
private_ENR_18_19 = private_ENR_18_19.set_index("Province")
private_ENR_18_19 = private_ENR_18_19.apply(pd.to_numeric, errors='coerce')

In [30]:
private_ENR_19_20.columns = ENR_COL
private_ENR_19_20 = private_ENR_19_20.iloc[1:-3].copy()
private_ENR_19_20 = private_ENR_19_20.set_index("Province")
private_ENR_19_20 = private_ENR_19_20.apply(pd.to_numeric, errors='coerce')

In [31]:
private_ENR_20_21.columns = ENR_COL
private_ENR_20_21 = private_ENR_20_21.iloc[1:-3].copy()
private_ENR_20_21 = private_ENR_20_21.set_index("Province")
private_ENR_20_21 = private_ENR_20_21.apply(pd.to_numeric, errors='coerce')

In [32]:
private_ENR_18_19.insert(0,'year',2018)
private_ENR_19_20.insert(0,'year',2019)
private_ENR_20_21.insert(0,'year',2020)

In [33]:
ENR = pd.concat([private_ENR_18_19,private_ENR_19_20,private_ENR_20_21])

In [34]:
flow_cal = pd.concat([GER,COM,ENR,combined_enroll],axis = 1)

In [35]:
flow_cal

,year,GAR(Total),GAR(F),GAR(M),NAR(Total),NAR(F),NAR(M),GER pri,GER L.Sec,GER U.Sec,GER pri(F),GER L.Sec(F),GER U.Sec(F),GER pri(M),GER L.Sec(M),GER U.Sec(M),NER pri,NER L.Sec,NER U.Sec,NER pri(F),NER L.Sec(F),NER U.Sec(F),NER pri(M),NER L.Sec(M),NER U.Sec(M),year,pri(Total),pri(F),pri(M),L.sec(Total),L.sec(F),L.sec(M),U.sec(Total),U.sec(F),U.sec(M),year,Number of school,pre(Total),pre(F),pri(Total),pri(F),L.sec(Total),L.sec(F),U.sec(Total),U.sec(F),Total(Total),Total(F),Female pri(%),Female all(%),Year,Grade_1_Enrollment_Total,Grade_1_Enrollment_Girl,Grade_1_Repeaters_Total,Grade_1_Repeaters_Girl,Grade_2_Enrollment_Total,Grade_2_Enrollment_Girl,Grade_2_Repeaters_Total,Grade_2_Repeaters_Girl,Grade_3_Enrollment_Total,Grade_3_Enrollment_Girl,Grade_3_Repeaters_Total,Grade_3_Repeaters_Girl,Grade_4_Enrollment_Total,Grade_4_Enrollment_Girl,Grade_4_Repeaters_Total,Grade_4_Repeaters_Girl,Grade_5_Enrollment_Total,Grade_5_Enrollment_Girl,Grade_5_Repeaters_Total,Grade_5_Repeaters_Girl,Grade_6_Enrollment_Total,Grade_6_Enrollment_Girl,Grade_6_Repeaters_Total,Grade_6_Repeaters_Girl,Grade_7_Enrollment_Total,Grade_7_Enrollment_Girl,Grade_7_Repeaters_Total,Grade_7_Repeaters_Girl,Grade_8_Enrollment_Total,Grade_8_Enrollment_Girl,Grade_8_Repeaters_Total,Grade_8_Repeaters_Girl,Grade_9_Enrollment_Total,Grade_9_Enrollment_Girl,Grade_9_Repeaters_Total,Grade_9_Repeaters_Girl,Grade_10_Enrollment_Total,Grade_10_Enrollment_Girl,Grade_10_Repeaters_Total,Grade_10_Repeaters_Girl,Grade_11_Enrollment_Total,Grade_11_Enrollment_Girl,Grade_11_Repeaters_Total,Grade_11_Repeaters_Girl,Grade_12_Enrollment_Total,Grade_12_Enrollment_Girl,Grade_12_Repeaters_Total,Grade_12_Repeaters_Girl,Total_Enrollment_Total,Total_Enrollment_Girl,Total_Repeaters_Total,Total_Repeaters_Girl
Province,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,19.2,18.4,20.1,12.1,11.9,12.2,14.1,5.1,2.1,13.9,5.1,2.2,14.2,5.0,2.0,12.6,4.1,1.8,12.5,4.3,2.0,12.7,4.0,1.6,2018,8.2,8.8,7.7,3.9,4.2,3.7,1.7,1.7,1.7,2018,121.0,3393,1653,14332,6879,2917,1421,1300,658,21942,10611,48.0,48.4,2018,3270,1518,157,64,2972,1444,85,24,2594,1200,63,18,2166,1044,14,5,1829,901,16,6,1501,772,12,4,1229,602,12,4,904,421,9,3,784,398,16,4,515,276,0,0,427,215,1,0,358,167,7,1,21942,10611,392,133
Battambang,2018,4.2,4.2,4.3,2.4,2.4,2.3,2.8,1.9,1.6,2.9,1.9,1.7,2.8,1.8,1.4,2.4,1.5,1.3,2.5,1.6,1.5,2.4,1.5,1.1,2018,2.0,2.1,2.0,1.8,1.9,1.7,1.3,1.5,1.2,2018,45.0,1273,605,4348,2139,1661,813,1522,814,8804,4371,49.2,49.6,2018,1028,487,3,0,828,386,8,0,689,352,10,4,620,310,5,1,608,312,7,0,575,292,4,2,545,277,14,2,547,250,9,1,569,286,14,2,540,285,5,3,529,284,2,1,453,245,17,7,8804,4371,98,23
Kampong Cham,2018,2.7,2.9,2.5,1.3,1.3,1.3,2.1,0.6,0.3,2.1,0.6,0.3,2.1,0.6,0.3,1.5,0.5,0.3,1.5,0.5,0.3,1.6,0.5,0.3,2018,1.3,1.3,1.3,0.5,0.5,0.4,0.2,0.3,0.2,2018,25.0,853,455,2532,1213,407,192,218,109,4010,1969,47.9,49.1,2018,545,278,32,10,528,253,27,13,455,215,17,6,337,145,8,3,381,177,0,0,286,145,4,4,159,72,2,1,142,66,2,0,106,54,0,0,89,48,1,0,71,33,0,0,58,28,0,0,4010,1969,93,37
Kampong Chhnang,2018,4.3,4.6,4.0,4.0,4.3,3.7,2.6,1.5,1.1,2.6,1.2,0.9,2.7,1.8,1.3,2.5,1.1,0.9,2.5,0.9,0.8,2.6,1.3,1.0,2018,1.5,1.4,1.7,1.5,1.2,1.8,1.3,1.1,1.4,2018,25.0,1532,736,1905,905,608,230,460,184,4505,2055,47.5,45.6,2018,499,261,3,1,426,191,3,1,246,121,0,0,278,124,0,0,254,119,0,0,202,89,0,0,211,80,5,1,182,68,0,0,215,82,1,0,119,36,2,0,165,75,0,0,176,73,1,1,4505,2055,15,4
Kampong Speu,2018,6.2,6.4,6.0,3.9,3.9,3.8,3.9,1.2,0.3,4.1,1.2,0.4,3.7,1.2,0.3,3.1,0.9,0.3,3.2,0.9,0.3,3.0,0.8,0.3,2018,2.4,2.7,2.1,1.0,1.2,0.8,0.2,0.2,0.2,2018,50.0,2917,1514,4276,2189,740,372,201,104,8134,4179,51.2,51.4,2018,1091,552,29,12,825,412,21,7,741,376,12,5,626,312,5,3,502,261,9,4,491,276,18,13,288,133,4,1,246,118,2,2,206,121,1,0,105,55,0,0,58,30,0,0,38,19,0,0,8134,4179,101,47
Kampong Thom,2018,1.6,1.3,1.8,0.8,0.5,1.1,1.1,0.4,0.2,1.0,0.3,0.2,1.2,0.4,0.2,0.9,0.2,0.1,0.8,0.2,0.1,1.0,0.2,0.1,2018,0.7,0.7,0.7,0.3,0.2,0.3,0.1,0.1,0.1

In [36]:
# ============================================================
# FEATURE ENGINEERING: Graduation, Repetition, Dropout Rates
# ============================================================
# Assumes you already have these DataFrames from your cleaning:
#   combined_enroll  — Province index, Year col, Grade_1..12 enrollment+repeaters
#   combined_df      — completion rates (pri, L.sec, U.sec)
# ============================================================

"""
Returns a new DataFrame with one row per Province×Year containing:
REPETITION RATES (per grade, primary = G1-G6, secondary = G7-G12)
DROPOUT RATES (between grades)
dropout_G{n}_to_G{n+1} — students lost between consecutive grades
                         = 1 - (E_{n+1} / (E_n - R_n))
                         where R_n = repeaters in grade n
graduation_pri       — completion rate primary   (from completion_df if given)
graduation_lsec      — completion rate L.sec     (from completion_df if given)
graduation_usec      — completion rate U.sec     (from completion_df if given)
"""

'\nReturns a new DataFrame with one row per Province×Year containing:\nREPETITION RATES (per grade, primary = G1-G6, secondary = G7-G12)\nDROPOUT RATES (between grades)\ndropout_G{n}_to_G{n+1} — students lost between consecutive grades\n                         = 1 - (E_{n+1} / (E_n - R_n))\n                         where R_n = repeaters in grade n\ngraduation_pri       — completion rate primary   (from completion_df if given)\ngraduation_lsec      — completion rate L.sec     (from completion_df if given)\ngraduation_usec      — completion rate U.sec     (from completion_df if given)\n'

In [37]:
def engineer_rates(combined_enroll: pd.DataFrame, completion_df: pd.DataFrame = None) -> pd.DataFrame:
    # Debug: print initial row count
    # print(f"[DEBUG] Input rows: {len(combined_enroll)}")

    df = combined_enroll.copy()

    # ── 1. NORMALIZE ENROLLMENT DATA ──────────────────────────────────
    # Ensure Province and Year are columns
    if "Province" not in df.columns:
        df = df.reset_index()                     # index becomes a column (named 'index')
        # Rename that column to 'Province' if it's the first column
        if "Province" not in df.columns and len(df.columns) > 0:
            df = df.rename(columns={df.columns[0]: "Province"})

    # Standardize column casing for 'Year'
    df.columns = [c.capitalize() if c.lower() == 'year' else c for c in df.columns]

    # If Province still not found, try renaming first column as fallback
    if "Province" not in df.columns and len(df.columns) > 0:
        df = df.rename(columns={df.columns[0]: "Province"})

    # Debug: check column names after normalisation
    # print(f"[DEBUG] Columns after normalisation: {df.columns.tolist()}")

    # Drop aggregate rows (exact match only)
    # Adjust this list to match the exact aggregate names in your data
    agg_values = ["Whole Kingdom", "Urban", "Rural", "nan"]
    # Use .str.strip() to handle possible leading/trailing spaces
    mask = df["Province"].astype(str).str.strip().isin(agg_values)
    df = df[~mask].copy()

    # Debug: print number of rows after dropping aggregates
    # print(f"[DEBUG] Rows after dropping aggregates: {len(df)}")

    # Coerce numeric
    for col in df.columns:
        if col not in ("Province", "Year"):
            df[col] = pd.to_numeric(df[col], errors="coerce")

    out = df[["Province", "Year"]].copy()

    # ── 2. REPETITION RATES ───────────────────────────────────────────
    grades = list(range(1, 13))
    for g in grades:
        e_col, r_col = f"Grade_{g}_Enrollment_Total", f"Grade_{g}_Repeaters_Total"
        eg_col, rg_col = f"Grade_{g}_Enrollment_Girl", f"Grade_{g}_Repeaters_Girl"

        if e_col in df.columns and r_col in df.columns:
            out[f"rep_rate_G{g}"] = (df[r_col] / df[e_col].replace(0, np.nan) * 100).round(2)
        if eg_col in df.columns and rg_col in df.columns:
            out[f"rep_rate_G{g}_girl"] = (df[rg_col] / df[eg_col].replace(0, np.nan) * 100).round(2)

    pri_rep_cols = [f"rep_rate_G{g}" for g in range(1, 7) if f"rep_rate_G{g}" in out.columns]
    sec_rep_cols = [f"rep_rate_G{g}" for g in range(7, 13) if f"rep_rate_G{g}" in out.columns]
    out["rep_rate_pri_avg"] = out[pri_rep_cols].mean(axis=1).round(2)
    out["rep_rate_sec_avg"] = out[sec_rep_cols].mean(axis=1).round(2)

    # ── 3. DROPOUT RATES (CONSECUTIVE) ───────────────────────────────
    for g in range(1, 12):
        e_n, r_n, e_n1 = f"Grade_{g}_Enrollment_Total", f"Grade_{g}_Repeaters_Total", f"Grade_{g+1}_Enrollment_Total"
        if all(c in df.columns for c in [e_n, r_n, e_n1]):
            promoters = (df[e_n] - df[r_n]).clip(lower=0)
            dropout_rate = (1 - df[e_n1] / promoters.replace(0, np.nan)) * 100
            out[f"dropout_G{g}_to_G{g+1}"] = dropout_rate.clip(lower=0, upper=100).round(2)

    # ── 4. STAGE DROPOUT & SURVIVAL ──────────────────────────────────
    def get_survival(df_src, start, end, gender="Total"):
        surv = pd.Series(1.0, index=df_src.index)
        for g in range(start, end):
            en, rn, en1 = f"Grade_{g}_Enrollment_{gender}", f"Grade_{g}_Repeaters_{gender}", f"Grade_{g+1}_Enrollment_{gender}"
            if all(c in df_src.columns for c in [en, rn, en1]):
                prom = (df_src[en] - df_src[rn]).clip(lower=0)
                surv *= (df_src[en1] / prom.replace(0, np.nan)).clip(0, 1)
        return surv

    out["dropout_pri"] = ((1 - get_survival(df, 1, 6)) * 100).round(2)
    out["dropout_lsec"] = ((1 - get_survival(df, 7, 9)) * 100).round(2)
    out["dropout_usec"] = ((1 - get_survival(df, 10, 12)) * 100).round(2)

    # ── 5. MERGE GRADUATION DATA ─────────────────────────────────────
    if completion_df is not None:
        comp = completion_df.copy()

        # Ensure 'Year' and 'Province' are columns in the completion data
        if "Province" not in comp.columns:
            comp = comp.reset_index()
            if "Province" not in comp.columns and len(comp.columns) > 0:
                comp = comp.rename(columns={comp.columns[0]: "Province"})

        # Standardise column casing for 'Year'
        comp.columns = [c.capitalize() if c.lower() == 'year' else c for c in comp.columns]

        # Debug: check columns in comp
        # print(f"[DEBUG] Completion columns: {comp.columns.tolist()}")

        if "Year" in comp.columns and "Province" in comp.columns:
            # Clean summary rows in completion data (exact match only)
            agg_values = ["Whole Kingdom", "Urban", "Rural", "nan"]
            mask_comp = comp["Province"].astype(str).str.strip().isin(agg_values)
            comp = comp[~mask_comp].copy()

            # Identify columns for merge (look for "total" and stage indicators)
            pri_col = next((c for c in comp.columns if "pri" in c.lower() and "total" in c.lower()), None)
            lsec_col = next((c for c in comp.columns if "l.sec" in c.lower() and "total" in c.lower()), None)
            usec_col = next((c for c in comp.columns if "u.sec" in c.lower() and "total" in c.lower()), None)

            merge_cols = ["Province", "Year"]
            rename_map = {}
            if pri_col:
                merge_cols.append(pri_col)
                rename_map[pri_col] = "graduation_pri"
            if lsec_col:
                merge_cols.append(lsec_col)
                rename_map[lsec_col] = "graduation_lsec"
            if usec_col:
                merge_cols.append(usec_col)
                rename_map[usec_col] = "graduation_usec"

            if len(merge_cols) > 2:
                comp_sub = comp[merge_cols].rename(columns=rename_map)
                # Ensure merge keys have the same dtype
                # (convert to string if necessary, but better to keep as is)
                out = out.merge(comp_sub, on=["Province", "Year"], how="left")
                # Debug: check row count after merge
                # print(f"[DEBUG] Rows after merge: {len(out)}")

    # Final clean-up
    out = out.reset_index(drop=True)

    # Debug: final row count
    # print(f"[DEBUG] Final output rows: {len(out)}")
    return out

In [38]:
combined_pri_flow=engineer_rates(combined_enroll,COM)

In [39]:
combined_pri_flow

,Province,Year,rep_rate_G1,rep_rate_G1_girl,rep_rate_G2,rep_rate_G2_girl,rep_rate_G3,rep_rate_G3_girl,rep_rate_G4,rep_rate_G4_girl,rep_rate_G5,rep_rate_G5_girl,rep_rate_G6,rep_rate_G6_girl,rep_rate_G7,rep_rate_G7_girl,rep_rate_G8,rep_rate_G8_girl,rep_rate_G9,rep_rate_G9_girl,rep_rate_G10,rep_rate_G10_girl,rep_rate_G11,rep_rate_G11_girl,rep_rate_G12,rep_rate_G12_girl,rep_rate_pri_avg,rep_rate_sec_avg,dropout_G1_to_G2,dropout_G2_to_G3,dropout_G3_to_G4,dropout_G4_to_G5,dropout_G5_to_G6,dropout_G6_to_G7,dropout_G7_to_G8,dropout_G8_to_G9,dropout_G9_to_G10,dropout_G10_to_G11,dropout_G11_to_G12,dropout_pri,dropout_lsec,dropout_usec,graduation_pri,graduation_lsec,graduation_usec
0,Banteay Meanchey,2018,4.8,4.22,2.86,1.66,2.43,1.5,0.65,0.48,0.87,0.67,0.8,0.52,0.98,0.66,1.0,0.71,2.04,1.01,0.0,0.0,0.23,0.0,1.96,0.6,2.07,1.03,4.53,10.15,14.42,15.01,17.21,17.46,25.72,12.4,32.94,17.09,15.96,48.34,34.93,30.32,8.2,3.9,1.7
1,Battambang,2018,0.29,0.0,0.97,0.0,1.45,1.14,0.81,0.32,1.15,0.0,0.7,0.68,2.57,0.72,1.65,0.4,2.46,0.7,0.93,1.05,0.38,0.35,3.75,2.86,0.9,1.96,19.22,15.98,8.69,1.14,4.33,4.55,0.0,0.0,2.7,1.12,14.04,41.38,0.0,15.01,2.0,1.8,1.3
2,Kampong Cham,2018,5.87,3.6,5.11,5.14,3.74,2.79,2.37,2.07,0.0,0.0,1.4,2.76,1.26,1.39,1.41,0.0,0.0,0.0,1.12,0.0,0.0,0.0,0.0,0.0,3.08,0.63,0.0,9.18,23.06,0.0,24.93,43.62,9.55,24.29,16.04,19.32,18.31,47.55,31.52,34.09,1.3,0.5,0.2
3,Kampong Chhnang,2018,0.6,0.38,0.7,0.52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.37,1.25,0.0,0.0,0.47,0.0,1.68,0.0,0.0,0.0,0.57,1.37,0.22,0.85,14.11,41.84,0.0,8.63,20.47,0.0,11.65,0.0,44.39,0.0,0.0,63.71,11.65,0.0,1.5,1.5,1.3
4,Kampong Speu,2018,2.66,2.17,2.55,1.7,1.62,1.33,0.8,0.96,1.79,1.53,3.67,4.71,1.39,0.75,0.81,1.69,0.49,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.18,0.45,22.32,7.84,14.13,19.16,0.41,39.11,13.38,15.57,48.78,44.76,34.48,50.5,26.87,63.81,2.4,1.0,0.2
5,Kampong Thom,2018,0.43,0.0,0.52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.36,4.55,0.0,0.0,0.0,0.0,0.0,0.0,0.16,0.89,17.67,0.0,29.33,0.0,21.19,40.34,0.0,32.53,0.0,47.27,51.72,54.15,32.53,74.55,0.7,0.3,0.1
6,Kampot,2018,2.31,2.78,2.2,2.06,3.42,3.67,1.25,1.2,1.15,0.81,0.72,0.0,1.24,1.63,0.0,0.0,0.26,0.0,0.0,0.0,0.0,0.0,6.65,6.32,1.84,1.36,39.29,27.1,5.9,11.44,18.87,0.0,30.47,1.03,3.66,17.62,0.0,70.08,31.18,17.62,3.6,2.3,2.0
7,Kandal,2018,6.49,4.79,7.94,5.52,5.72,4.12,4.44,3.2,2.18,1.18,3.41,1.38,0.79,0.68,0.81,1.65,0.57,0.44,0.74,1.1,0.79,0.0,2.86,4.19,5.03,1.09,20.97,17.33,6.3,8.63,17.24,0.0,17.44,14.47,22.18,5.94,7.16,53.71,29.39,12.68,2.2,1.8,1.1
8,Kep,2018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,64.71,11.76,0.0,21.15,41.46,100.0,<NA>,<NA>,<NA>,<NA>,<NA>,85.63,<NA>,<NA>,3.7,0.0,0.0
9,Koh Kong,2018,1.64,0.55,3.47,2.04,0.25,0.0,0.8,0.0,1.06,2.19,0.0,0.0,0.0,0.0,0.85,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,1.2,0.21,19.86,29.08,5.33,23.78,15.77,41.7,14.6,32.76,78.21,100.0,<NA>,65.46,42.57,<NA>,6.4,2.0,0.0


In [40]:
def restructure_to_cohort_flow(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Setup the basic structure
    new_df = pd.DataFrame()
    
    # Preserve Province and Year (ensure they exist as columns)
    temp_df = df.reset_index() if "Province" not in df.columns else df.copy()
    new_df["Province"] = temp_df["Province"]
    new_df["Year"] = temp_df["Year"]

    # 2. Loop through Grades 1 to 12
    for g in range(1, 13):
        # Source column names from your current DF
        rep_col = f"rep_rate_G{g}"
        
        # Determine Dropout source: 
        # Your DF uses 'dropout_G{n}_to_G{n+1}', Grade 12 uses 'dropout_usec' or similar
        if g < 12:
            drop_src = f"dropout_G{g}_to_G{g+1}"
        else:
            drop_src = "dropout_usec" # Proxy for Grade 12

        # A: REPETITION (Direct Copy)
        if rep_col in temp_df.columns:
            new_df[f"Grade_{g}_Repetition"] = pd.to_numeric(temp_df[rep_col], errors='coerce').round(1)
        else:
            new_df[f"Grade_{g}_Repetition"] = np.nan

        # B: PROMOTION & DROPOUT
        # Logic: Promotion = 100 - (Dropout + Repetition)
        if drop_src in temp_df.columns:
            drop_val = pd.to_numeric(temp_df[drop_src], errors='coerce')
            rep_val = new_df[f"Grade_{g}_Repetition"]
            
            # Create the Promotion column
            new_df[f"Grade_{g}_Promotion"] = (100 - (drop_val + rep_val)).clip(lower=0).round(1)
            # Create the Dropout column
            new_df[f"Grade_{g}_Dropout"] = drop_val.round(1)
        else:
            new_df[f"Grade_{g}_Promotion"] = np.nan
            new_df[f"Grade_{g}_Dropout"] = np.nan

    # 3. Final Column Ordering (Grade 1 Promotion, Rep, Drop... Province, Year)
    ordered_cols = []
    for g in range(1, 13):
        ordered_cols.extend([f"Grade_{g}_Promotion", f"Grade_{g}_Repetition", f"Grade_{g}_Dropout"])
    
    ordered_cols.extend(["Province", "Year"])
    
    # Return only the restructured columns, dropping all old average/graduation columns
    return new_df[ordered_cols]

# Usage:
# restructured_df = restructure_to_cohort_flow(your_original_df)

In [41]:
combined_pri_flow=restructure_to_cohort_flow(combined_pri_flow)

In [42]:
combined_pri_flow

,Grade_1_Promotion,Grade_1_Repetition,Grade_1_Dropout,Grade_2_Promotion,Grade_2_Repetition,Grade_2_Dropout,Grade_3_Promotion,Grade_3_Repetition,Grade_3_Dropout,Grade_4_Promotion,Grade_4_Repetition,Grade_4_Dropout,Grade_5_Promotion,Grade_5_Repetition,Grade_5_Dropout,Grade_6_Promotion,Grade_6_Repetition,Grade_6_Dropout,Grade_7_Promotion,Grade_7_Repetition,Grade_7_Dropout,Grade_8_Promotion,Grade_8_Repetition,Grade_8_Dropout,Grade_9_Promotion,Grade_9_Repetition,Grade_9_Dropout,Grade_10_Promotion,Grade_10_Repetition,Grade_10_Dropout,Grade_11_Promotion,Grade_11_Repetition,Grade_11_Dropout,Grade_12_Promotion,Grade_12_Repetition,Grade_12_Dropout,Province,Year
0,90.7,4.8,4.5,87.0,2.9,10.2,83.2,2.4,14.4,84.4,0.6,15.0,81.9,0.9,17.2,81.7,0.8,17.5,73.3,1.0,25.7,86.6,1.0,12.4,65.1,2.0,32.9,82.9,0.0,17.1,83.8,0.2,16.0,67.7,2.0,30.3,Banteay Meanchey,2018
1,80.5,0.3,19.2,83.0,1.0,16.0,89.9,1.4,8.7,98.1,0.8,1.1,94.5,1.2,4.3,94.8,0.7,4.6,97.4,2.6,0.0,98.4,1.6,0.0,94.8,2.5,2.7,98.0,0.9,1.1,85.6,0.4,14.0,81.2,3.8,15.0,Battambang,2018
2,94.1,5.9,0.0,85.7,5.1,9.2,73.2,3.7,23.1,97.6,2.4,0.0,75.1,0.0,24.9,55.0,1.4,43.6,89.2,1.3,9.6,74.3,1.4,24.3,84.0,0.0,16.0,79.6,1.1,19.3,81.7,0.0,18.3,65.9,0.0,34.1,Kampong Cham,2018
3,85.3,0.6,14.1,57.5,0.7,41.8,100.0,0.0,0.0,91.4,0.0,8.6,79.5,0.0,20.5,100.0,0.0,0.0,86.0,2.4,11.6,100.0,0.0,0.0,55.1,0.5,44.4,98.3,1.7,0.0,100.0,0.0,0.0,99.4,0.6,0.0,Kampong Chhnang,2018
4,75.0,2.7,22.3,89.6,2.6,7.8,84.3,1.6,14.1,80.0,0.8,19.2,97.8,1.8,0.4,57.2,3.7,39.1,85.2,1.4,13.4,83.6,0.8,15.6,50.7,0.5,48.8,55.2,0.0,44.8,65.5,0.0,34.5,36.2,0.0,63.8,Kampong Speu,2018
5,81.9,0.4,17.7,99.5,0.5,0.0,70.7,0.0,29.3,100.0,0.0,0.0,78.8,0.0,21.2,59.7,0.0,40.3,100.0,0.0,0.0,67.5,0.0,32.5,94.6,5.4,0.0,52.7,0.0,47.3,48.3,0.0,51.7,25.5,0.0,74.6,Kampong Thom,2018
6,58.4,2.3,39.3,70.7,2.2,27.1,90.7,3.4,5.9,87.4,1.2,11.4,79.9,1.2,18.9,99.3,0.7,0.0,68.3,1.2,30.5,99.0,0.0,1.0,96.0,0.3,3.7,82.4,0.0,17.6,100.0,0.0,0.0,75.8,6.6,17.6,Kampot,2018
7,72.5,6.5,21.0,74.8,7.9,17.3,88.0,5.7,6.3,87.0,4.4,8.6,80.6,2.2,17.2,96.6,3.4,0.0,81.8,0.8,17.4,84.7,0.8,14.5,77.2,0.6,22.2,93.4,0.7,5.9,92.0,0.8,7.2,84.4,2.9,12.7,Kandal,2018
8,35.3,0.0,64.7,88.2,0.0,11.8,100.0,0.0,0.0,78.8,0.0,21.2,58.5,0.0,41.5,0.0,0.0,100.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Kep,2018
9,78.5,1.6,19.9,67.4,3.5,29.1,94.5,0.2,5.3,75.4,0.8,23.8,83.1,1.1,15.8,58.3,0.0,41.7,85.4,0.0,14.6,66.4,0.8,32.8,21.8,0.0,78.2,0.0,0.0,100.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Koh Kong,2018


In [43]:
combined_pri_flow['Year'].value_counts()

Year
2018    25
2019    25
2020    25
Name: count, dtype: int64

In [44]:

combined_df = combined_df.reset_index().set_index(['Province', 'year'])
combined_enroll = combined_enroll.reset_index().set_index(['Province', 'Year'])
combined_staff = combined_staff.reset_index().set_index(['Province', 'Year'])
combined_pri_flow = combined_pri_flow.reset_index().set_index(['Province', 'Year'])

# Now try the concat again
private_school_2018_2020 = pd.concat([
    combined_df, 
    combined_enroll, 
    combined_staff, 
    combined_pri_flow
], axis=1)

In [45]:
rename_map = {
    "Total_Primary": "T_Primary",
    "Total_LSec": "T_LSec",
    "Total_USec": "T_USec",
    "Total_Graduate": "T_Graduate",
    "Female_Primary": "NT_Primary", # Check if this logic holds for your data!
    "NoPedTrain_Primary": "NoPed_Primary",
}

# 1. Rename the columns
private_school_2018_2020 = private_school_2018_2020.rename(columns=rename_map)

# 2. Drop the useless building columns that aren't in your target
cols_to_drop = ["index"]
private_school_2018_2020 = private_school_2018_2020.drop(columns=[c for c in cols_to_drop if c in private_school_2018_2020.columns])

In [46]:
private_school_2018_2020

,,Schools,Classes,Enrollment_Total,Enrollment_Girl,TeachStaff_Total,TeachStaff_Female,NonTeachStaff_Total,NonTeachStaff_Female,Govern_Staff,Foreigner_Staff_Total,Foreigner_Staff_Female,Buildings,Rooms,Classrooms,Grade_1_Enrollment_Total,Grade_1_Enrollment_Girl,Grade_1_Repeaters_Total,Grade_1_Repeaters_Girl,Grade_2_Enrollment_Total,Grade_2_Enrollment_Girl,Grade_2_Repeaters_Total,Grade_2_Repeaters_Girl,Grade_3_Enrollment_Total,Grade_3_Enrollment_Girl,Grade_3_Repeaters_Total,Grade_3_Repeaters_Girl,Grade_4_Enrollment_Total,Grade_4_Enrollment_Girl,Grade_4_Repeaters_Total,Grade_4_Repeaters_Girl,Grade_5_Enrollment_Total,Grade_5_Enrollment_Girl,Grade_5_Repeaters_Total,Grade_5_Repeaters_Girl,Grade_6_Enrollment_Total,Grade_6_Enrollment_Girl,Grade_6_Repeaters_Total,Grade_6_Repeaters_Girl,Grade_7_Enrollment_Total,Grade_7_Enrollment_Girl,Grade_7_Repeaters_Total,Grade_7_Repeaters_Girl,Grade_8_Enrollment_Total,Grade_8_Enrollment_Girl,Grade_8_Repeaters_Total,Grade_8_Repeaters_Girl,Grade_9_Enrollment_Total,Grade_9_Enrollment_Girl,Grade_9_Repeaters_Total,Grade_9_Repeaters_Girl,Grade_10_Enrollment_Total,Grade_10_Enrollment_Girl,Grade_10_Repeaters_Total,Grade_10_Repeaters_Girl,Grade_11_Enrollment_Total,Grade_11_Enrollment_Girl,Grade_11_Repeaters_Total,Grade_11_Repeaters_Girl,Grade_12_Enrollment_Total,Grade_12_Enrollment_Girl,Grade_12_Repeaters_Total,Grade_12_Repeaters_Girl,Total_Enrollment_Total,Total_Enrollment_Girl,Total_Repeaters_Total,Total_Repeaters_Girl,T_Primary,T_LSec,T_USec,T_Graduate,Total_PostGrad,Total_PhD,NT_Primary,Female_LSec,Female_USec,Female_Graduate,Female_PostGrad,Female_PhD,NoPed_Primary,NoPedTrain_LSec,NoPedTrain_USec,NoPedTrain_Graduate,NoPedTrain_PostGrad,NoPedTrain_PhD,Grade_1_Promotion,Grade_1_Repetition,Grade_1_Dropout,Grade_2_Promotion,Grade_2_Repetition,Grade_2_Dropout,Grade_3_Promotion,Grade_3_Repetition,Grade_3_Dropout,Grade_4_Promotion,Grade_4_Repetition,Grade_4_Dropout,Grade_5_Promotion,Grade_5_Repetition,Grade_5_Dropout,Grade_6_Promotion,Grade_6_Repetition,Grade_6_Dropout,Grade_7_Promotion,Grade_7_Repetition,Grade_7_Dropout,Grade_8_Promotion,Grade_8_Repetition,Grade_8_Dropout,Grade_9_Promotion,Grade_9_Repetition,Grade_9_Dropout,Grade_10_Promotion,Grade_10_Repetition,Grade_10_Dropout,Grade_11_Promotion,Grade_11_Repetition,Grade_11_Dropout,Grade_12_Promotion,Grade_12_Repetition,Grade_12_Dropout
Province,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Banteay Meanchey,2018,121,894,21942,10611,1053,580,252,113,269,49,25,132,1382,703,3270,1518,157,64,2972,1444,85,24,2594,1200,63,18,2166,1044,14,5,1829,901,16,6,1501,772,12,4,1229,602,12,4,904,421,9,3,784,398,16,4,515,276,0,0,427,215,1,0,358,167,7,1,21942,10611,392,133,0,38,430,554,31,0,0,29,265,280,6,0,0,8,50,51,1,0,90.7,4.8,4.5,87.0,2.9,10.2,83.2,2.4,14.4,84.4,0.6,15.0,81.9,0.9,17.2,81.7,0.8,17.5,73.3,1.0,25.7,86.6,1.0,12.4,65.1,2.0,32.9,82.9,0.0,17.1,83.8,0.2,16.0,67.7,2.0,30.3
Battambang,2018,45,406,8804,4371,779,488,165,82,552,38,27,57,632,361,1028,487,3,0,828,386,8,0,689,352,10,4,620,310,5,1,608,312,7,0,575,292,4,2,545,277,14,2,547,250,9,1,569,286,14,2,540,285,5,3,529,284,2,1,453,245,17,7,8804,4371,98,23,12,31,228,467,40,1,12,19,169,274,14,0,0,1,5,8,0,0,80.5,0.3,19.2,83.0,1.0,16.0,89.9,1.4,8.7,98.1,0.8,1.1,94.5,1.2,4.3,94.8,0.7,4.6,97.4,2.6,0.0,98.4,1.6,0.0,94.8,2.5,2.7,98.0,0.9,1.1,85.6,0.4,14.0,81.2,3.8,15.0
Kampong Cham,2018,25,158,4010,1969,214,130,34,14,54,7,6,52,295,140,545,278,32,10,528,253,27,13,455,215,17,6,337,145,8,3,381,177,0,0,286,145,4,4,159,72,2,1,142,66,2,0,106,54,0,0,89,48,1,0,71,33,0,0,58,28,0,0,4010,1969,93,37,5,23,74,103,9,0,1,17,59,53,0,0,0,0,3,8,0,0,94.1,5.9,0.0,85.7,5.1,9.2,73.2,3.7,23.1,97.6,2.4,0.0,75.1,0.0,24.9,55.0,1.4,43.6,89.2,1.3,9.6,74.3,1.4,24.3,84.0,0.0,16.0,79.6,1.1,19.3,81.7,0.0,18.3,65.9,0.0,34.1
Kampong Chhnang,2018,25,190,4505,2055,356,242,87,46,132,13,9,51,326,183,499,261,3,1,426,191,3,1,246,121,0,0,278,124,0,0,254,119,0,0,202,89,0,0,211,80,5,1,182,68,0,

In [47]:
cols = private_school_2018_2020.columns.tolist()

# Print 5 columns per line
for i in range(0, len(cols), 5):
    print(f"{i:3}: {', '.join(cols[i:i+5])}")

  0: Schools, Classes, Enrollment_Total, Enrollment_Girl, TeachStaff_Total
  5: TeachStaff_Female, NonTeachStaff_Total, NonTeachStaff_Female, Govern_Staff, Foreigner_Staff_Total
 10: Foreigner_Staff_Female, Buildings, Rooms, Classrooms, Grade_1_Enrollment_Total
 15: Grade_1_Enrollment_Girl, Grade_1_Repeaters_Total, Grade_1_Repeaters_Girl, Grade_2_Enrollment_Total, Grade_2_Enrollment_Girl
 20: Grade_2_Repeaters_Total, Grade_2_Repeaters_Girl, Grade_3_Enrollment_Total, Grade_3_Enrollment_Girl, Grade_3_Repeaters_Total
 25: Grade_3_Repeaters_Girl, Grade_4_Enrollment_Total, Grade_4_Enrollment_Girl, Grade_4_Repeaters_Total, Grade_4_Repeaters_Girl
 30: Grade_5_Enrollment_Total, Grade_5_Enrollment_Girl, Grade_5_Repeaters_Total, Grade_5_Repeaters_Girl, Grade_6_Enrollment_Total
 35: Grade_6_Enrollment_Girl, Grade_6_Repeaters_Total, Grade_6_Repeaters_Girl, Grade_7_Enrollment_Total, Grade_7_Enrollment_Girl
 40: Grade_7_Repeaters_Total, Grade_7_Repeaters_Girl, Grade_8_Enrollment_Total, Grade_8_Enrol

In [48]:
#private_school_2018_2020.to_csv('private_cleaned.csv')